# Individual Tax‑Saving Advisor (RAG w/ Milvus Lite and Base Model w/ GPT-oss-20b)

This notebook:
1) Clones repo: `https://github.com/imnotmomo/6895mid.git`
2) Loads corpus from folders (`text/`, `meta/`, `eval/`)
3) Uses a **strong embedding model** (`BAAI/bge-base-en-v1.5` by default)
4) **Drops & recreates** Milvus collection (reindex)
5) Indexes chunks into Milvus
6) Runs **RAG evaluation** (Hit@k, MRR)

Requirement:
1) Hugging face token
2) Cloudflare Tunnel Token

## 0) Install dependencies

In [1]:
!pip -q install -U "pymilvus[milvus-lite]" sentence-transformers trafilatura pypdf
!pip -q install -U "pandas==2.2.2"
# %pip uninstall -y transformers tokenizers
%pip install --no-cache-dir \
"transformers==4.57.1" \
"tokenizers>=0.21.0" \
accelerate \
peft \
trl \
safetensors \
bitsandbytes \
"triton>=3.4.0" \
kernels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 31.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 210.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 885.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 224.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 656.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys
import importlib
sys.path.insert(0, "/usr/local/lib/python3.12/dist-packages")
importlib.invalidate_caches()
import transformers
import inspect
print("version:", transformers.__version__)
print("path:", inspect.getfile(transformers))

version: 4.57.1
path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py


## 1) Clone files from GitHub repo

In [3]:
!rm -rf 6895mid
!git clone https://github.com/imnotmomo/6895mid.git
!ls -lah 6895mid | head -n 120

Cloning into '6895mid'...
remote: Enumerating objects: 815, done.
remote: Counting objects: 100% (815/815), done.
remote: Compressing objects: 100% (555/555), done.
remote: Total 815 (delta 349), reused 722 (delta 259), pack-reused 0 (from 0)
Receiving objects: 100% (815/815), 13.22 MiB | 41.03 MiB/s, done.
Resolving deltas: 100% (349/349), done.
total 904K
drwxr-xr-x 8 root root 4.0K Mar 11 05:53 .
drwxr-xr-x 1 root root 4.0K Mar 11 05:53 ..
-rw-r--r-- 1 root root 828K Mar 11 05:53 6895mid_finalversion-17_fixed.ipynb
drwxr-xr-x 2 root root 4.0K Mar 11 05:53 eval
drwxr-xr-x 8 root root 4.0K Mar 11 05:53 .git
drwxr-xr-x 2 root root  16K Mar 11 05:53 html
drwxr-xr-x 2 root root  20K Mar 11 05:53 meta
drwxr-xr-x 2 root root 4.0K Mar 11 05:53 pdf
drwxr-xr-x 2 root root  20K Mar 11 05:53 text


## 2) Locate corpus folders
Repo must contain `text/` and `meta/`. `eval/` is used for evaluation.

In [4]:
from pathlib import Path

repo_root = Path('6895mid')
assert repo_root.exists(), 'Repo folder not found. Did git clone run?'

corpus_root = None
candidates = [repo_root] + [p for p in repo_root.rglob('*') if p.is_dir()]
for p in candidates:
    if (p / 'text').is_dir() and (p / 'meta').is_dir():
        corpus_root = p
        break

assert corpus_root is not None, 'Could not find corpus root containing text/ and meta/.'

text_dir = corpus_root / 'text'
meta_dir = corpus_root / 'meta'
eval_dir = corpus_root / 'eval'

print('✅ corpus_root:', corpus_root)
print('Text files:', len(list(text_dir.glob('*.txt'))))
print('Meta files:', len(list(meta_dir.glob('*.json'))))
print('Eval exists:', eval_dir.exists())
if eval_dir.exists():
    print('Eval files:', [p.name for p in sorted(eval_dir.glob('*'))])

✅ corpus_root: 6895mid
Text files: 184
Meta files: 192
Eval exists: True
Eval files: ['benchmark.json', 'rag_eval_set.json']


## 3) Load embedding model

In [5]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = 'BAAI/bge-base-en-v1.5'

embedder = SentenceTransformer(EMBED_MODEL_NAME)
DIM = embedder.get_sentence_embedding_dimension()
print('Embedding model:', EMBED_MODEL_NAME)
print('Embedding dim:', DIM)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: BAAI/bge-base-en-v1.5
Embedding dim: 768


## 4) Helpers: chunking, categorization, URL metadata, INT64-safe ids

In [6]:
import json, hashlib, re

def chunk_text(text: str, max_chars=1500, min_chars=60, overlap_chars=200):
    """
    Larger chunks (1500 chars) with 200-char overlap so tax rules stay intact.
    Lower min_chars (60) keeps short but rule-dense paragraphs that the old
    min_chars=120 would have dropped.  Expect ~350+ chunks instead of ~209.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    chunks = []
    cur = []
    cur_len = 0

    for p in paras:
        if cur and cur_len + len(p) + 2 > max_chars:
            block = "\n\n".join(cur).strip()
            if len(block) >= min_chars:
                chunks.append(block)
            # --- overlap: keep last paragraph(s) up to overlap_chars ---
            overlap = []
            ov_len = 0
            for prev_p in reversed(cur):
                if ov_len + len(prev_p) + 2 > overlap_chars:
                    break
                overlap.insert(0, prev_p)
                ov_len += len(prev_p) + 2
            cur = overlap + [p]
            cur_len = sum(len(x) for x in cur) + 2 * (len(cur) - 1)
        else:
            cur.append(p)
            cur_len += len(p) + 2

    if cur:
        block = "\n\n".join(cur).strip()
        if len(block) >= min_chars:
            chunks.append(block)

    if not chunks and text.strip():
        chunks = [text[:max_chars].strip()]

    return chunks

def categorize(doc_id: str) -> str:
    """Kept for metadata only — search() no longer filters on this."""
    d = doc_id.lower()
    if 'instructions' in d:
        return 'form_instruction'
    if 'publications__p' in d or d.endswith('__p17'):
        return 'publication'
    if any(k in d for k in ['credits', 'deductions', 'taxtopics', 'filing', 'individuals', 'retirement-plans']):
        return 'guidance'
    return 'other'

url_map = {}
for p in meta_dir.glob('*.json'):
    try:
        j = json.loads(p.read_text())
        if 'url' in j:
            url_map[p.stem] = j['url']
    except Exception:
        pass

print('URL metadata loaded for', len(url_map), 'documents')

INT64_MAX = (1 << 63) - 1
def int_id(doc_id: str, chunk_i: int, chunk: str) -> int:
    h = hashlib.sha1(f"{doc_id}|{chunk_i}|{chunk}".encode('utf-8')).hexdigest()[:16]
    return int(h, 16) & INT64_MAX

URL metadata loaded for 192 documents


## 5) Milvus Lite (via MilvusClient) — DROP & RECREATE (reindex)

In [7]:
from pymilvus import MilvusClient

DB_PATH = './milvus_irs_demo.db'
COLLECTION = 'irs_individual'

client = MilvusClient(DB_PATH)
existing = client.list_collections()
print('Existing collections:', existing)

if COLLECTION in existing:
    client.drop_collection(COLLECTION)
    print('🧹 Dropped collection:', COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    dimension=DIM,
    metric_type='COSINE'
)
print('✅ Recreated collection:', COLLECTION, 'DIM=', DIM)

Existing collections: []
✅ Recreated collection: irs_individual DIM= 768


## 6) Index corpus into Milvus Lite (chunk → embed → insert)

In [8]:
txt_files = sorted(text_dir.glob('*.txt'))
print('Found', len(txt_files), 'text files')

BATCH = 128
batch = []
inserted = 0

def flush_batch(batch):
    global inserted
    data = []
    for x in batch:
        data.append({
            'id': x['id'],
            'vector': x['vector'],
            'doc': x['doc'],
            'category': x['category'],
            'url': x['url'],
            'text': x['text'],
        })
    client.insert(collection_name=COLLECTION, data=data)
    inserted += len(batch)

for f in txt_files:
    doc_id = f.stem
    raw = f.read_text(encoding='utf-8', errors='ignore').strip()
    if not raw:
        continue
    chunks = chunk_text(raw)
    if not chunks:
        continue
    embs = embedder.encode(chunks, normalize_embeddings=True)
    cat = categorize(doc_id)
    url = url_map.get(doc_id, '')
    for i, (chunk, vec) in enumerate(zip(chunks, embs)):
        batch.append({
            'id': int_id(doc_id, i, chunk),
            'vector': vec.tolist(),
            'doc': doc_id,
            'category': cat,
            'url': url,
            'text': chunk,
        })
        if len(batch) >= BATCH:
            flush_batch(batch)
            batch.clear()

if batch:
    flush_batch(batch)

print('✅ Inserted chunks:', inserted)

Found 184 text files
✅ Inserted chunks: 4227


## 7) Search helper (RAG retrieval)

In [9]:
import re
from sentence_transformers import CrossEncoder

# --- Cross-encoder reranker for much better retrieval precision ---
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
print("Cross-encoder reranker loaded.")

def keyword_overlap(query: str, text: str) -> float:
    q_tokens = set(re.findall(r"[a-z0-9]+", query.lower()))
    q_tokens = {t for t in q_tokens if len(t) > 2}
    t_tokens = set(re.findall(r"[a-z0-9]+", text.lower()))
    if not q_tokens:
        return 0.0
    return len(q_tokens & t_tokens) / len(q_tokens)

def diversify_hits(hits, top_k=10, per_doc=5):
    out = []
    seen = {}
    for h in hits:
        doc = h["metadata"].get("doc", "")
        if seen.get(doc, 0) >= per_doc:
            continue
        out.append(h)
        seen[doc] = seen.get(doc, 0) + 1
        if len(out) >= top_k:
            break
    return out

def search(query: str, top_k=10, overfetch=40):
    """
    FIXED: No category filter.  The old filter='category in [publication,
    form_instruction, guidance]' silently dropped every doc tagged 'other'
    — which included passive-activity, at-risk, property-transactions, and
    many more.  Now we search ALL chunks and let the cross-encoder decide
    what's relevant.
    """
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()

    res = client.search(
        collection_name=COLLECTION,
        data=[qv],
        limit=overfetch,
        # NO filter — search all categories
        output_fields=['doc', 'category', 'url', 'text']
    )

    hits = [{'score': float(h['distance']), 'metadata': h['entity']} for h in res[0]]

    # --- Stage 1: cross-encoder reranking ---
    pairs = [(query, h["metadata"].get("text", "")[:512]) for h in hits]
    if pairs:
        ce_scores = reranker.predict(pairs)
        for h, ce_s in zip(hits, ce_scores):
            h["rank_score"] = float(ce_s)
    else:
        for h in hits:
            h["rank_score"] = h["score"]

    # --- Stage 2: small keyword bonus (secondary signal) ---
    for h in hits:
        bonus = 0.5 * keyword_overlap(query, h["metadata"].get("text", "")[:1000])
        h["rank_score"] += bonus

    hits.sort(key=lambda x: x["rank_score"], reverse=True)
    return diversify_hits(hits, top_k=top_k, per_doc=3)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder reranker loaded.


## 8) RAG evaluation (Hit@k + MRR)

In [10]:
import pandas as pd
import json

rag_eval_path = eval_dir / 'rag_eval_set.json'
assert rag_eval_path.exists(), f'Missing: {rag_eval_path}'
rag_cases = json.loads(rag_eval_path.read_text())
print('Loaded RAG eval cases:', len(rag_cases))

def doc_matches_expected(doc_id: str, expected_tokens):
    d = (doc_id or '').lower()
    return any(tok.lower() in d for tok in expected_tokens)

def eval_rag(top_k=5):
    rows = []
    hits = 0
    rr_sum = 0.0
    for c in rag_cases:
        q = c['query']
        expected = c.get('expected_sources_any', [])
        matches = search(q, top_k=top_k)
        docs = [m['metadata'].get('doc','') for m in matches]
        first_rank = None
        for i, d in enumerate(docs, start=1):
            if doc_matches_expected(d, expected):
                first_rank = i
                break
        hit = first_rank is not None
        if hit:
            hits += 1
            rr_sum += 1.0 / first_rank
        rows.append({'id': c.get('id',''), 'hit': hit, 'first_rank': first_rank, 'top_docs': docs})
    hit_at_k = hits / max(1, len(rag_cases))
    mrr = rr_sum / max(1, len(rag_cases))
    return hit_at_k, mrr, pd.DataFrame(rows)

hit5, mrr5, df = eval_rag(top_k=5)
print(f'Hit@5: {hit5:.1%}')
print(f'MRR@5: {mrr5:.3f}')
df[['id','hit','first_rank']]

Loaded RAG eval cases: 5
Hit@5: 80.0%
MRR@5: 0.600


,id,hit,first_rank
0,rag_edu_aotc,True,1.0
1,rag_dep_ctc,True,1.0
2,rag_income_taxable,False,NaN
3,rag_ira_deduction,True,2.0
4,rag_capital_gains,True,2.0


# LLM Part

## 0) Set up Hugging Face token


In [11]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print('HF_TOKEN set, logged in')
else:
    print('HF_TOKEN not set.')

HF_TOKEN set, logged in


## 1) Base model

In [12]:
#!rm -rf ~/.cache/huggingface

In [13]:
# import os
# os._exit(0)

In [14]:
!pip install triton>=3.5.0

In [15]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "openai/gpt-oss-20b"

print('Loading base model:', BASE_MODEL)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)



model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype="auto",
    device_map='auto',
    trust_remote_code=True,
)

model.eval()
print('✅ Model loaded')
print(model.hf_device_map)

Loading base model: openai/gpt-oss-20b


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

_topk_forward.py: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

✅ Model loaded
{'': 0}


## 2) Prompt builder + RAG answer

In [16]:
!pip -q install openai-harmony

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.8 MB/s eta 0:00:00


In [17]:
import re

def renumber_used_sources(data, cite_map):
    used = data.get("sources_used", [])
    used = [int(x) for x in used if str(x).isdigit()]
    if not used:
        return data, cite_map

    remap = {old: new for new, old in enumerate(sorted(set(used)), start=1)}

    def remap_text_citations(text):
        if not isinstance(text, str):
            return text
        return re.sub(
            r"\^(\d+)",
            lambda m: f"^{remap.get(int(m.group(1)), int(m.group(1)))}",
            text,
        )

    data["answer_bullets"] = [
        remap_text_citations(b) for b in data.get("answer_bullets", [])
    ]

    for case in data.get("scenario_cases", []):
        case["bullets"] = [
            remap_text_citations(b) for b in case.get("bullets", [])
        ]

    data["sources_used"] = [remap[x] for x in used if x in remap]

    new_cite_map = []
    for c in cite_map:
        old_n = c["n"]
        if old_n in remap:
            c2 = dict(c)
            c2["n"] = remap[old_n]
            new_cite_map.append(c2)

    new_cite_map.sort(key=lambda x: x["n"])
    return data, new_cite_map

In [18]:

import json
import re
import torch

TOP_K = 15
CONTEXT_MAX_CHARS = 5000
MAX_SOURCES = 8
MAX_NEW_TOKENS = 1200

SYSTEM_PROMPT = """
You are an IRS-grounded individual tax-saving advisor.
Return ONLY valid JSON.
Do not output markdown.
Do not output code fences.
Do not output explanations outside JSON.
Do not ask follow-up questions outside the JSON fields.
"""

FIELD_HELP = {
    "filing status": "Tell me whether you are single, married filing jointly, married filing separately, or head of household.",
    "deduction type": "Tell me whether you plan to take the standard deduction or itemize. If you are not sure, say 'not sure' and I can compare both.",
    "MAGI": "Tell me your modified adjusted gross income if known.",
    "IRA type": "Tell me whether this is a traditional IRA or Roth IRA.",
    "Workplace retirement plan coverage": "Tell me whether you are covered by a workplace retirement plan such as a 401(k).",
    "Compensation": "Tell me your taxable compensation or earned income.",
    "dependents": "Tell me how many dependents you are claiming, if any.",
    "tax year": "Tell me which tax year you want the answer for.",
    "business type": "Tell me whether you are self-employed, a sole proprietor, an independent contractor, or a W-2 employee.",
    "exclusive use of home office": "Tell me whether you use the home office space exclusively and regularly for business (not for personal use).",
    "home office area/percentage": "Tell me the square footage of your home office, or the percentage of your home used for business.",
    "holding period": "Tell me how long you held the asset before selling (e.g. less than 1 year, or more than 1 year).",
    "number of children": "Tell me how many qualifying children you have.",
    "ages of children": "Tell me the ages of your children.",
    "enrollment status": "Tell me whether the student is enrolled at least half-time in a degree program."
}

def build_context_and_numbered_sources(matches, max_chars=CONTEXT_MAX_CHARS, max_sources=MAX_SOURCES):
    ctx_blocks = []
    cite_map = []
    doc_to_n = {}
    total = 0
    next_n = 1

    for m in matches:
        md = m.get("metadata", {})
        doc = md.get("doc", "")
        url = md.get("url", "")
        text = md.get("text", "")
        if not text:
            continue

        if doc not in doc_to_n:
            if len(cite_map) >= max_sources:
                continue
            doc_to_n[doc] = next_n
            snippet = " ".join(text.split()[:18])
            cite_map.append({
                "n": next_n,
                "doc": doc,
                "url": url,
                "snippet": snippet
            })
            next_n += 1

        n = doc_to_n[doc]
        block = f"SOURCE {n}: {doc}\n{text}"

        if total + len(block) > max_chars:
            continue

        ctx_blocks.append(block)
        total += len(block)

    sources_block = "\n".join(
        [f"{c['n']}. {c['url'] if c['url'] else c['doc']}" for c in cite_map]
    )
    return "\n\n---\n\n".join(ctx_blocks), sources_block, cite_map


def build_prompt_json(question: str, context: str, cite_map, state_text="", force_proceed=False):
    source_ids = [c["n"] for c in cite_map]

    schema = {
        "mode": "final_or_missing_or_scenario",
        "answer_bullets": ["Short bullet with citation like ^1"],
        "missing_fields": ["field name"],
        "scenario_cases": [
            {
                "label": "Scenario label",
                "bullets": ["Possible outcome under this assumption.^1"]
            }
        ],
        "sources_used": [1]
    }

    scenario_rule = ""
    if force_proceed:
        scenario_rule = """
Scenario mode is ON:
- If required information is missing, do NOT determine the user's actual eligibility.
- Instead return mode = "scenario".
- Provide the main possible outcomes under clearly labeled assumptions.
- The user must decide which scenario matches their situation.
"""

    state_block = state_text if state_text and state_text.strip() else "(none)"

    return f"""
Use ONLY the provided IRS sources.

Return ONLY valid JSON.
Do not output markdown.
Do not output code fences.
Do not output analysis.

Rules:
1. If enough information is available, return mode = "final".
2. If information is missing and scenario mode is OFF, return mode = "missing".
3. If information is missing and scenario mode is ON, return mode = "scenario".
4. In "answer_bullets", give the direct answer.
5. In "missing_fields", list only missing required fields.
6. In "scenario_cases", list possible outcomes only when mode = "scenario".
7. Use only these citation ids in bullets: {source_ids}
8. In "sources_used", include only source numbers actually used.
9. Be concise: at most 3 answer bullets, at most 5 missing fields, at most 3 scenario cases.

Question classification (CRITICAL — follow strictly):
10. EDUCATIONAL questions ask how a rule works in general, e.g. "What is a Roth IRA?", "How does the standard deduction work?", "What are the capital gains tax brackets?". For these, return mode = "final" with a general IRS-grounded explanation.
11. PERSONAL ELIGIBILITY questions ask about the user's own situation, e.g. "Can I claim ...", "Am I eligible for ...", "How much can I deduct ...", "Should I ...", "Do I qualify for ...". For these, you MUST check whether the user has provided enough personal facts (filing status, income/MAGI, dependents, etc.) to give a specific answer. If key facts are missing from "Known user facts" and the question text, return mode = "missing".
12. When in doubt between educational and personal, treat the question as personal if it uses first-person pronouns ("I", "my", "me") or asks about eligibility, qualification, or amounts.

No-assumption rule (CRITICAL):
12a. For personal eligibility questions, NEVER assume or infer personal facts that the user has not stated. Do not guess filing status, income level, employment type, or any other personal detail. If a required fact is missing, return mode = "missing" — do not fill in the gap with a likely or common value.
12b. Even if the question implies a likely scenario (e.g. "Can I claim a home office deduction if I work for myself?" implies self-employment), you must still ask for ALL other required facts that are not explicitly stated (e.g. exclusive use, area/percentage).
12c. Only when force_proceed / scenario mode is ON may you make assumptions — and in that case, return mode = "scenario" with clearly labeled assumptions, never mode = "final".

Missing fields rules:
13. Do not treat the topic of the question itself as a missing field. "missing_fields" must contain only concrete missing tax facts, not paraphrases of the question.
14. Examples of valid missing fields: filing status, MAGI, IRA type, workplace retirement plan coverage, compensation, business type, exclusive use of home office, number of dependents. Examples of invalid missing fields: "IRA deduction rules", "deduction changes for filing jointly", "tax effects".
15. Common required facts by topic:
    - Home office deduction: business type (self-employed vs employee), exclusive/regular use, home office area/percentage.
    - IRA deduction: IRA type, workplace retirement plan coverage, MAGI (if covered by plan), filing status.
    - Capital gains: holding period, filing status, taxable income/MAGI.
    - Child tax credit: number of children, ages, filing status, income.
    - Education credits: filing status, MAGI, enrollment status.
    If ANY of the required facts for the topic are missing from the user's question and known facts, return mode = "missing".

Clarification handling:
16. If the user is asking what a missing field means, asking how to answer it, or asking for examples, treat that as a clarification request.
17. For a clarification request, return mode = "final" and explain the missing field in plain language with examples of valid answers.
18. Do NOT return mode = "missing" when the user's current request is asking for explanation or examples of a previously requested field.
19. If prior conversation context identifies which missing field the user is asking about, use that context.

IRA rules:
- Roth IRA contributions are not deductible.
- If not covered by a workplace retirement plan, MAGI does not limit traditional IRA deductibility.
- If covered by a workplace retirement plan, deductibility may be limited by MAGI.
- Compensation affects the deductible limit, not basic deductibility eligibility.
- Do not refuse solely because compensation is unknown unless the user asks for the exact deductible amount.

{scenario_rule}

Output schema:
{json.dumps(schema, indent=2)}

Known user facts:
{state_block}

User question:
{question}

IRS sources:
{context}
""".strip()


def generate_json(prompt: str, max_new_tokens=MAX_NEW_TOKENS):
    """
    GPT-OSS-20B generation using chat template.
    We decode raw generated text and let the JSON parser extract the object.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][input_len:]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=False)

    for tok in [
        "<|start|>", "<|message|>", "<|end|>",
        "<|channel|>analysis", "<|channel|>final",
        "<|assistant|>", "<|return|>"
    ]:
        raw = raw.replace(tok, "")

    return raw.strip()


def parse_json_output(raw: str):
    raw = raw.strip()

    if not raw:
        raise ValueError("Empty model output.")

    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except Exception:
        pass

    decoder = json.JSONDecoder()
    start = raw.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in model output.\nRaw output:\n{raw[:1000]}")
    obj, end = decoder.raw_decode(raw[start:])
    return obj


def normalize_output(data: dict):
    """
    Make field names/user-facing labels cleaner.
    """
    field_map = {
        "magI": "MAGI",
        "magi": "MAGI",
        "agi": "AGI",
        "ira_type": "IRA type",
        "workplace_plan_covered": "Workplace retirement plan coverage",
        "filing_status": "Filing status",
        "compensation": "Compensation"
    }

    data["missing_fields"] = [
        field_map.get(f, f) for f in data.get("missing_fields", [])
    ]
    return data



def render_answer_from_json(data, cite_map):
    mode = data.get("mode", "missing")
    answer_lines = []

    if mode == "final":
        answer_lines.append("Answer:")
        for b in data.get("answer_bullets", []):
            answer_lines.append(f"- {b}")

    elif mode == "missing":
        answer_lines.append("[MISSING_INFO]")
        answer_lines.append("Answer:")
        answer_lines.append("I need a few facts before I can determine the result.")
        answer_lines.append("Please provide:")

        missing = data.get("missing_fields", [])
        for f in missing:
            help_text = FIELD_HELP.get(f, "Please provide this information.")
            answer_lines.append(f"- {f}: {help_text}")

        if missing:
            answer_lines.append("")
            answer_lines.append("You can reply with just the missing items above, or use Force Proceed to get scenario-based outcomes.")

    elif mode == "scenario":
        answer_lines.append("Answer:")
        answer_lines.append("Insufficient information to determine the actual result.")
        answer_lines.append("Possible outcomes:")

        for i, case in enumerate(data.get("scenario_cases", []), start=1):
            answer_lines.append(f"{i}. {case.get('label', 'Scenario')}")
            for b in case.get("bullets", []):
                answer_lines.append(f"   - {b}")

        missing = data.get("missing_fields", [])
        if missing:
            answer_lines.append("Missing fields needed to determine the actual case:")
            for f in missing:
                help_text = FIELD_HELP.get(f, "Please provide this information.")
                answer_lines.append(f"- {f}: {help_text}")

    used = set(str(x) for x in data.get("sources_used", []))
    source_lines = ["", "Sources:"]
    valid_ns = set()
    for c in cite_map:
        if str(c["n"]) in used:
            url_or_doc = c["url"] if c.get("url") else c["doc"]
            source_lines.append(f"{c['n']}. {url_or_doc}")
            valid_ns.add(str(c["n"]))

    if len(source_lines) == 2:
        for c in cite_map[:2]:
            url_or_doc = c["url"] if c.get("url") else c["doc"]
            source_lines.append(f"{c['n']}. {url_or_doc}")
            valid_ns.add(str(c["n"]))

    fallback_n = max(valid_ns) if valid_ns else None

    cleaned_answer = []
    for line in answer_lines:
        new_line = re.sub(
            r"\^(\d+)",
            lambda m: m.group(0) if m.group(1) in valid_ns else (f"^{fallback_n}" if fallback_n else ""),
            line
        )
        new_line = re.sub(r"\^(\d+)(,\^(\1))+", r"^\1", new_line)
        cleaned_answer.append(new_line)

    return "\n".join(cleaned_answer + source_lines).strip()


def answer_with_rag_json(question: str, top_k=10, state_text="", force_proceed=False):
    matches = search(question, top_k=top_k)
    context, _, cite_map = build_context_and_numbered_sources(
        matches,
        max_chars=CONTEXT_MAX_CHARS,
        max_sources=MAX_SOURCES,
    )

    prompt = build_prompt_json(
        question=question,
        context=context,
        cite_map=cite_map,
        state_text=state_text,
        force_proceed=force_proceed,
    )

    raw = generate_json(prompt, max_new_tokens=MAX_NEW_TOKENS)

    try:
        data = normalize_output(parse_json_output(raw))
        data, cite_map = renumber_used_sources(data, cite_map)
        return render_answer_from_json(data, cite_map)
    except Exception:
        retry_prompt = prompt + "\n\nIMPORTANT: Output ONLY the JSON object now. Start with {"
        raw = generate_json(retry_prompt, max_new_tokens=MAX_NEW_TOKENS)
        data = normalize_output(parse_json_output(raw))
        data, cite_map = renumber_used_sources(data, cite_map)
        return render_answer_from_json(data, cite_map)

## 3) Example test


In [19]:
question = "Can I deduct IRA contributions?"
print(answer_with_rag_json(question, top_k=TOP_K))


[MISSING_INFO]
Answer:
I need a few facts before I can determine the result.
Please provide:
- filing status: Tell me whether you are single, married filing jointly, married filing separately, or head of household.
- MAGI: Tell me your modified adjusted gross income if known.
- employer retirement plan coverage: Please provide this information.
- IRA type: Tell me whether this is a traditional IRA or Roth IRA.

You can reply with just the missing items above, or use Force Proceed to get scenario-based outcomes.

Sources:
1. https://www.irs.gov/publications/p590a
2. https://www.irs.gov/publications/p554


In [20]:
q = "If my AGI is 100000, passive income is 10000, passive loss is 45000, and basis and at-risk limits cap it at 12000, what is my deductible amount and AGI?"
print(answer_with_rag_json(q, top_k=TOP_K))

Answer:
- You can deduct up to $12,000 of passive loss, which is the amount limited by your basis and at‑risk rules, and this reduces your AGI to $88,000.^1
- The $12,000 deduction is the maximum you may claim against your nonpassive income for the year, as the passive loss is fully limited by the basis and at‑risk cap.^2

Sources:
1. https://www.irs.gov/publications/p925
2. https://www.irs.gov/instructions/i8582


## 4) Conversation window example


In [21]:
# 1) Simple conversation window
conv = {
    "history": [],
    "state": {}
}

def user_append(conv, text):
    conv["history"].append({"role": "user", "content": text})

def ai_append(conv, text):
    conv["history"].append({"role": "assistant", "content": text})

def state_to_text(conv):
    if not conv["state"]:
        return ""
    lines = []
    for k, v in conv["state"].items():
        if v is not None:
            lines.append(f"{k}: {v}")
    return "\n".join(lines)

In [22]:
# 2) One conversation turn
def answer_turn(conv, user_msg, top_k=10):
    user_append(conv, user_msg)

    state_text = state_to_text(conv)
    if state_text:
        question = f"""Known user facts:
{state_text}

User question:
{user_msg}"""
    else:
        question = user_msg

    ans = answer_with_rag_json(question, top_k=top_k, state_text=state_text)
    ai_append(conv, ans)
    return ans

In [23]:
# 3) First turn: user asks, model should ask for more information
q1 = "Can I deduct IRA contributions? What affects eligibility?"
a1 = answer_turn(conv, q1, top_k=10)

print("USER:")
print(q1)
print("\nASSISTANT:")
print(a1)

USER:
Can I deduct IRA contributions? What affects eligibility?

ASSISTANT:
[MISSING_INFO]
Answer:
I need a few facts before I can determine the result.
Please provide:
- filing status: Tell me whether you are single, married filing jointly, married filing separately, or head of household.
- MAGI: Tell me your modified adjusted gross income if known.
- employer retirement plan coverage: Please provide this information.

You can reply with just the missing items above, or use Force Proceed to get scenario-based outcomes.

Sources:
1. https://www.irs.gov/retirement-plans/plan-participant-employee/retirement-savings-contributions-credit-savers-credit
2. https://www.irs.gov/publications/p590a


In [24]:
# 4) User provides the missing facts; store them in conversation state
conv["state"].update({
    "ira_type": "traditional",
    "age": 27,
    "workplace_plan_covered": False,
    "magi": 85000,
    "filing_status": "single"
})

q2 = "Traditional IRA. Age 27. Not covered by a workplace retirement plan. MAGI 85000. Filing status single."
a2 = answer_turn(conv, q2, top_k=10)

print(a2)

Answer:
- You can deduct the full amount of your traditional IRA contribution up to the annual limit, because you are not covered by a workplace retirement plan and your MAGI does not limit deductibility.^1

Sources:
1. https://www.irs.gov/publications/p590b


# API access (FastAPI) + Cloudflare exposure

In [25]:
!pip -q install -U fastapi uvicorn nest-asyncio "requests==2.32.4"

In [26]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="IRS Tax Advisor API")

origins = [
    "http://localhost:3000",
    "http://127.0.0.1:3000",
    "https://your-frontend.example.com",
]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    question: str
    top_k: int = 10
    force_proceed: bool = False

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/chat")
def chat(req: ChatRequest):
    if not req.question or not req.question.strip():
        raise HTTPException(status_code=400, detail="question is required")

    answer = answer_with_rag_json(
        question=req.question,
        top_k=req.top_k,
        state_text="",
        force_proceed=req.force_proceed,
    )
    return {"answer": answer}


In [27]:
import time
import requests
import nest_asyncio, threading, uvicorn

nest_asyncio.apply()

server_config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(server_config)

def run_api():
    server.run()

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
print("✅ API server starting on http://0.0.0.0:8000")

# wait until API is ready
for _ in range(20):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("✅ API is ready")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("API did not become ready in time")

INFO:     Started server process [12295]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ API server starting on http://0.0.0.0:8000
INFO:     127.0.0.1:59120 - "GET /health HTTP/1.1" 200 OK
✅ API is ready


In [28]:
import requests

print(requests.get("http://127.0.0.1:8000/health", timeout=10).json())

payload = {
    "question": "Can I deduct IRA contributions? What affects eligibility?",
    "top_k": 10,
    "force_proceed": False,
}
r = requests.post("http://127.0.0.1:8000/chat", json=payload, timeout=180)
print(r.status_code)
print(r.json()["answer"][:1200])


INFO:     127.0.0.1:59126 - "GET /health HTTP/1.1" 200 OK
{'ok': True}
INFO:     127.0.0.1:59132 - "POST /chat HTTP/1.1" 200 OK
200
[MISSING_INFO]
Answer:
I need a few facts before I can determine the result.
Please provide:
- filing status: Tell me whether you are single, married filing jointly, married filing separately, or head of household.
- MAGI: Tell me your modified adjusted gross income if known.
- employer retirement plan coverage: Please provide this information.

You can reply with just the missing items above, or use Force Proceed to get scenario-based outcomes.

Sources:
1. https://www.irs.gov/retirement-plans/plan-participant-employee/retirement-savings-contributions-credit-savers-credit
2. https://www.irs.gov/publications/p590a


## 7) Cloudflare Tunnel for API Exposure


In [29]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version


cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


In [30]:
import os
import subprocess
import threading

token = userdata.get('CLOUDFLARE_TUNNEL_TOKEN')

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "run", "--protocol", "http2", "--token", token],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def reader():
    for line in proc.stdout:
        print(line, end="")

threading.Thread(target=reader, daemon=True).start()
print("Started named tunnel for your fixed domain")

Started named tunnel for your fixed domain


# Evaluation

In [31]:
import json
from pathlib import Path
import pandas as pd

repo_root = Path("6895mid/eval")
bench_path = repo_root / "benchmark.json"

cases = json.loads(bench_path.read_text())
print("Loaded cases:", len(cases))

Loaded cases: 28


In [ ]:
import re
from difflib import SequenceMatcher
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd

CONTEXT_MAX_CHARS_MCQ = 6000
MCQ_DEBUG = {}  # stores raw outputs for wrong-answer debugging

def _norm(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def _extract_mcq_choice(answer_text: str, options: dict) -> str | None:
    """
    Extract MCQ letter from model output.
    Uses findall + last match to avoid first-mention bias.
    """
    if not answer_text:
        return None

    up = answer_text.upper()
    valid_keys = set(options.keys())

    # --- Priority 1: find the LAST "ANSWER: X" pattern ---
    for pat in [
        r"\bANSWER\s*[:\-]?\s*([A-D])\b",
        r"\bCHOICE\s*[:\-]?\s*([A-D])\b",
        r"\bTHE\s+ANSWER\s+IS\s+([A-D])\b",
        r"\bCORRECT\s+ANSWER\s*[:\-]?\s*([A-D])\b",
    ]:
        matches = re.findall(pat, up)
        if matches and matches[-1] in valid_keys:
            return matches[-1]

    # --- Priority 2: "Option X" or "(X)" pattern, last match ---
    for pat in [
        r"\bOPTION\s+([A-D])\b",
        r"\(([A-D])\)",
    ]:
        matches = re.findall(pat, up)
        if matches and matches[-1] in valid_keys:
            return matches[-1]

    # --- Priority 3: bare letter on its own line ---
    for pat in [
        r"^\s*([A-D])\s*$",
        r"^\s*([A-D])\.\s",
    ]:
        m = re.search(pat, up, re.MULTILINE)
        if m and m.group(1) in valid_keys:
            return m.group(1)

    # --- Priority 4: exact option text match (last occurrence) ---
    ans_n = _norm(answer_text)
    last_match = None
    for k, v in options.items():
        v_n = _norm(v)
        if v_n and v_n in ans_n:
            pos = ans_n.rfind(v_n)
            if last_match is None or pos > last_match[1]:
                last_match = (k, pos)
    if last_match:
        return last_match[0]

    # --- Priority 5: fuzzy match with HIGH confidence threshold ---
    best_k, best_score = None, -1.0
    ans_tokens = set(ans_n.split())

    for k, v in options.items():
        v_n = _norm(v)
        v_tokens = set(v_n.split())
        seq_score = SequenceMatcher(None, ans_n, v_n).ratio()
        tok_score = len(ans_tokens & v_tokens) / max(1, len(v_tokens))
        score = max(seq_score, tok_score)
        if score > best_score:
            best_k, best_score = k, score

    if best_score >= 0.6:
        return best_k

    return None


def _generate_mcq(prompt: str, max_new_tokens: int = 768,
                   temperature: float = 0.0) -> str:
    """Shared generation helper for MCQ evaluation.
    temperature=0.0 -> greedy; >0 -> sampling for self-consistency."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a U.S. tax expert taking a multiple-choice exam. "
                "Read all provided IRS reference material carefully. "
                "Think step by step, then choose exactly one answer from A, B, C, D.\n"
                "IMPORTANT: Consider every option equally. Do NOT default to option A.\n"
                "End your response with a line in this exact format:\n"
                "ANSWER: <LETTER>"
            ),
        },
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=6144
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen_kwargs.update(do_sample=False)

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    gen_ids = outputs[0][input_len:]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=False)

    for tok in [
        "<|start|>", "<|message|>", "<|end|>",
        "<|channel|>analysis", "<|channel|>final",
        "<|assistant|>", "<|return|>"
    ]:
        raw = raw.replace(tok, "")

    return raw.strip()


# -- Phase-1 helper: retrieve context (with query augmentation) --

def retrieve_mcq_context(question: str, options: dict = None,
                         top_k=15, max_chars=CONTEXT_MAX_CHARS_MCQ):
    """Return a plain-text context block from Milvus for a single MCQ question.
    Optionally augments the query with option keywords for better retrieval."""
    aug_query = question
    if options:
        opt_words = set()
        for v in options.values():
            for w in re.findall(r"[A-Za-z][a-z]{3,}", v):
                opt_words.add(w.lower())
        stop = {"the","and","for","that","this","with","from","are","was",
                "will","not","have","been","which","their","would","each",
                "than","into","only","also","more","other","none","above",
                "following","statement","correct","incorrect","true","false",
                "amount","total"}
        opt_kw = [w for w in opt_words if w not in stop]
        if opt_kw:
            aug_query = question + " " + " ".join(opt_kw[:8])

    matches = search(aug_query, top_k=top_k)
    context_parts = []
    total_chars = 0
    for m in matches:
        chunk = m.get("metadata", {}).get("text", "")
        doc = m.get("metadata", {}).get("doc", "")
        if not chunk:
            continue
        block = f"[{doc}]\n{chunk}"
        if total_chars + len(block) > max_chars:
            break
        context_parts.append(block)
        total_chars += len(block)
    return "\n\n---\n\n".join(context_parts) if context_parts else "(no sources retrieved)"


# -- Phase-3 helper: answer one MCQ with SELF-CONSISTENCY VOTING --

def _build_mcq_prompt(question: str, options: dict, context: str) -> str:
    """Build the MCQ prompt with computation scaffolding."""
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])

    return f"""Below are relevant excerpts from IRS publications and instructions.
Use them to answer the multiple-choice question that follows.

=== IRS REFERENCE MATERIAL ===
{context}
=== END REFERENCE MATERIAL ===

Question:
{question}

Options:
{options_text}

Instructions:
1. Identify what tax rule or computation the question is testing.
2. If the question involves numbers, show each calculation step explicitly:
   - Write out every arithmetic operation (e.g., "$50,000 + $40,000 = $90,000")
   - For basis/at-risk/passive loss questions, apply limits in this order:
     Step 1: Tax basis limit (contributions + share of all debt)
     Step 2: At-risk basis limit (contributions + recourse debt + qualified nonrecourse financing only)
     Step 3: Passive activity loss limit (passive losses only offset passive income)
   - For property transactions, identify: amount realized, adjusted basis, realized gain/loss, boot received, recognized gain/loss, new basis
   - For gift/estate tax: compare annual exclusion, lifetime exemption, split gifts, compute taxable gift
   - For like-kind exchanges: boot = cash received + debt relief - cash paid - debt assumed
   - For involuntary conversions: recognized gain = insurance proceeds - cost of replacement (if replacement costs less than proceeds); new basis = cost of replacement - deferred gain
   - For divorce transfers: Sec 1041 = no gain/loss recognized, transferee takes carryover basis
   - For passive loss allocation: allocate net disallowed loss proportionally among loss activities only (not income activities)
3. Eliminate any option that contradicts the IRS rules you found. Explain why each eliminated option is wrong.
4. After completing your analysis, compare your computed result to each remaining option.
5. Pick the option that matches your computation.
6. State your final answer on its own line in this format: ANSWER: <letter>

IMPORTANT: Do NOT default to option A. Consider every option equally. Show your elimination reasoning.

Think step by step:"""


def answer_mcq_with_rag(question: str, options: dict, context: str,
                        n_votes: int = 3) -> str | None:
    """
    Self-consistency MCQ evaluation:
    - 1 greedy decode + (n_votes-1) sampled decodes
    - Majority vote across all attempts
    - Falls back to retry prompt if no consensus
    """
    prompt = _build_mcq_prompt(question, options, context)
    votes = []
    raw_outputs = []

    # Vote 1: greedy
    raw = _generate_mcq(prompt, max_new_tokens=768, temperature=0.0)
    raw_outputs.append(("greedy", raw))
    c = _extract_mcq_choice(raw, options)
    if c:
        votes.append(c)

    # Votes 2..n: sampled with temperature
    for i in range(n_votes - 1):
        raw = _generate_mcq(prompt, max_new_tokens=768, temperature=0.5)
        raw_outputs.append((f"sample_{i+1}", raw))
        c = _extract_mcq_choice(raw, options)
        if c:
            votes.append(c)

    # Store debug info
    q_key = question[:80]
    MCQ_DEBUG[q_key] = {
        "votes": votes,
        "raw_outputs": [(tag, out[:300]) for tag, out in raw_outputs],
    }

    if votes:
        counter = Counter(votes)
        winner, count = counter.most_common(1)[0]
        return winner

    # All extractions failed - retry with direct prompt
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])
    retry_prompt = f"""Question:
{question}

Options:
{options_text}

Show your work step by step. Apply tax basis, at-risk, and passive loss limits in order if relevant.
For involuntary conversions: recognized gain = proceeds - replacement cost (only if replacement < proceeds).
Eliminate wrong options before choosing.
End with: ANSWER: <letter>

Step-by-step solution:"""

    raw = _generate_mcq(retry_prompt, max_new_tokens=512)
    return _extract_mcq_choice(raw, options)


print("MCQ helpers defined (with self-consistency voting, query augmentation, anti-A-bias)")

MCQ helpers defined: retrieve_mcq_context(), answer_mcq_with_rag(), _generate_mcq()


In [41]:
import gc, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from pymilvus import MilvusClient

# PHASE 1 - Retrieve context for ALL questions
print("Phase 1: Ensuring retrieval stack is loaded ...")

if 'embedder' not in dir() or embedder is None:
    embedder = SentenceTransformer(EMBED_MODEL_NAME)
    print("  Re-loaded embedder.")

if 'reranker' not in dir() or reranker is None:
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
    print("  Re-loaded reranker.")

try:
    client.close()
except Exception:
    pass
client = MilvusClient(DB_PATH)
print("  Milvus client (re)connected.")

print("Phase 1: Retrieving context for all questions ...")
case_contexts = {}
for c in tqdm(cases, desc="Retrieving"):
    ctx = retrieve_mcq_context(c["question"], options=c.get("options"), top_k=TOP_K)
    case_contexts[c["mcq_id"]] = ctx

print(f"Retrieved context for {len(case_contexts)} questions.")

# PHASE 2 - Close Milvus + free embedder/reranker memory
print("Phase 2: Closing Milvus and freeing retrieval memory ...")
try:
    client.close()
    print("  Milvus client closed.")
except Exception as e:
    print(f"  Milvus close warning: {e}")
client = None

try:
    del embedder
except NameError:
    pass
try:
    del reranker
except NameError:
    pass
embedder = None
reranker = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  Embedder + reranker freed.")

# PHASE 3 - Run LLM generation with self-consistency voting
print("Phase 3: Running LLM MCQ evaluation (3-vote self-consistency) ...")
rows = []
for c in tqdm(cases, desc="Evaluating MCQ (RAG)"):
    ctx = case_contexts[c["mcq_id"]]
    try:
        pred = answer_mcq_with_rag(c["question"], c["options"], ctx, n_votes=3)
        err = ""
    except Exception as e:
        pred = None
        err = str(e)

    rows.append({
        "topic": c.get("topic"),
        "module": c.get("module"),
        "topic_guess": c.get("topic_guess"),
        "mcq_id": c.get("mcq_id"),
        "question_number": c.get("question_number"),
        "question": c["question"],
        "gold": c["correct_answer"],
        "pred": pred,
        "correct": pred == c["correct_answer"],
        "error": err,
    })

print("Done - RAG MCQ evaluation complete.")

Phase 1: Ensuring retrieval stack is loaded …
  Re-loaded embedder.
  Re-loaded reranker.
  Milvus client (re)connected.
Phase 1: Retrieving context for all questions …


Retrieving:   0%|          | 0/28 [00:00<?, ?it/s]

Retrieved context for 28 questions.
Phase 2: Closing Milvus and freeing retrieval memory …
  Milvus client closed.
  Embedder + reranker freed.
Phase 3: Running LLM MCQ evaluation …


Evaluating MCQ (RAG):   0%|          | 0/28 [00:00<?, ?it/s]

INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     3.94.200.55:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.

### Result

In [42]:
import pandas as pd

df = pd.DataFrame(rows)

print("Overall accuracy:", df["correct"].mean())
print("Correct:", df["correct"].sum(), "/", len(df))
# print()
# print(df)

print()
print("By topic:")
print(
    df.groupby("topic", dropna=False)
      .agg(
          n=("correct", "size"),
          correct=("correct", "sum"),
          accuracy=("correct", "mean"),
      )
      .sort_values("topic")
)

Overall accuracy: 0.7142857142857143
Correct: 20 / 28

By topic:
                                   n  correct  accuracy
topic                                                  
gift_estate_and_charitable         7        6  0.857143
passive_activity_and_at_risk       7        4  0.571429
property_transactions              7        5  0.714286
retirement_and_financial_planning  7        5  0.714286


In [38]:
for topic in sorted(df["topic"].dropna().unique()):
    sub = df[(df["topic"] == topic) & (~df["correct"])]
    print("\n" + "=" * 80)
    print("Topic:", topic)
    print("Wrong:", len(sub))
    for _, r in sub.iterrows():
        print("\nQ:", r["question"][:120])
        print("Gold:", r["gold"], "Pred:", r["pred"])
        # Show voting debug info
        q_key = r["question"][:80]
        if q_key in MCQ_DEBUG:
            dbg = MCQ_DEBUG[q_key]
            print("  Votes:", dbg["votes"])
            for tag, out in dbg["raw_outputs"]:
                print(f"  [{tag}] ...{out[-200:]}")


Topic: gift_estate_and_charitable
Wrong: 1

Q: Lila Leon, an elderly taxpayer, purchased arecreational vehicle (RV) several years ago for $375,000. The RV is currently worth $250,000 andisexpected to beworth only about $200,000 when she dies. Lila intends to leave the RVto her grandson Luis when she dies and is considering gifting the RV to him now because she rarely uses it anymore.Assumethatthecurrentannual gifttaxexclusionis$19,000, the current gift and estate transfer tax rate is 40 percent, and Lila's cumulative taxable gifts have already exceeded her lifetimegift and estate transfer exclusion amount. By how much will thegift/estatetransfer tax on theRV increase or decrease ifLila gifts the RV to her grandson now rather than leaving the RV to him when she dies?
Gold: C Pred: B

Topic: passive_activity_and_at_risk
Wrong: 3

Q: In the current tax year, Charlie reported the following: Wages $ 50,000 Taxable interest 1,500 Dividends 3,000 Partnership X loss (15,000) Partnership Z inc

## Comparing with base LLM


In [43]:
import torch
from tqdm.auto import tqdm
import pandas as pd

def answer_mcq_raw_model(question: str, options: dict) -> str | None:
    """Raw model MCQ -- no retrieval, with anti-A-bias and self-consistency."""
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])

    prompt = f"""Answer this multiple-choice U.S. tax question.

Question:
{question}

Options:
{options_text}

Instructions:
1. Identify what tax rule or computation the question is testing.
2. If the question involves numbers, show each calculation step explicitly.
3. Eliminate any option that contradicts the tax rules. Explain why each is wrong.
4. After completing your analysis, compare your result to each option A, B, C, D.
5. Pick the option that matches your computation.

IMPORTANT: Consider every option equally. Do NOT default to A. Show elimination reasoning.

Think step by step:"""

    votes = []
    # Vote 1: greedy
    raw = _generate_mcq(prompt, max_new_tokens=768, temperature=0.0)
    c = _extract_mcq_choice(raw, options)
    if c:
        votes.append(c)

    # Votes 2-3: sampled
    for _ in range(2):
        raw = _generate_mcq(prompt, max_new_tokens=768, temperature=0.5)
        c = _extract_mcq_choice(raw, options)
        if c:
            votes.append(c)

    if votes:
        counter = Counter(votes)
        return counter.most_common(1)[0][0]

    # Retry with direct prompt
    retry_prompt = f"""Question:
{question}

Options:
{options_text}

Show your work step by step, then state your final answer.
ANSWER: """
    raw = _generate_mcq(retry_prompt, max_new_tokens=384)
    return _extract_mcq_choice(raw, options)

In [44]:
rows_raw = []

for c in tqdm(cases, desc="Evaluating RAW model"):
    pred = answer_mcq_raw_model(c["question"], c["options"])
    rows_raw.append({
        "topic": c.get("topic"),
        "module": c.get("module"),
        "mcq_id": c.get("mcq_id"),
        "question_number": c.get("question_number"),
        "question": c["question"],
        "gold": c["correct_answer"],
        "pred": pred,
        "correct": pred == c["correct_answer"],
    })

Evaluating RAW model:   0%|          | 0/28 [00:00<?, ?it/s]

INFO:     54.163.112.121:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK
INFO:     38.15.197.116:0 - "GET /health HTTP/1.1" 200 OK


In [45]:
df = pd.DataFrame(rows_raw)

print("Overall accuracy:", df["correct"].mean())
print("Correct:", df["correct"].sum(), "/", len(df))

print()
print("By topic:")
print(
    df.groupby("topic", dropna=False)
      .agg(
          n=("correct", "size"),
          correct=("correct", "sum"),
          accuracy=("correct", "mean"),
      )
      .sort_values("topic")
)

Overall accuracy: 0.6428571428571429
Correct: 18 / 28

By topic:
                                   n  correct  accuracy
topic                                                  
gift_estate_and_charitable         7        2  0.285714
passive_activity_and_at_risk       7        3  0.428571
property_transactions              7        6  0.857143
retirement_and_financial_planning  7        7  1.000000


## Reload Retrieval Stack

In [ ]:
import gc, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from pymilvus import MilvusClient

# Reload embedder
if 'embedder' not in dir() or embedder is None:
    embedder = SentenceTransformer(EMBED_MODEL_NAME)
    print("Embedder reloaded.")
else:
    print("Embedder already loaded.")

# Reload reranker
if 'reranker' not in dir() or reranker is None:
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
    print("Reranker reloaded.")
else:
    print("Reranker already loaded.")

# Reconnect Milvus
try:
    client.close()
except Exception:
    pass
client = MilvusClient(DB_PATH)
print("Milvus client reconnected.")

# Quick sanity check
cols = client.list_collections()
print(f"Collections: {cols}")
print("Retrieval stack is live — API and interactive cells will work.")